# Memoizing functions with joblib

Records hold results that deserve a type and a spec. Many analyses are just functions of other
results; `joblib.Memory` caches those by their arguments. `td.memory(location)` returns a
`joblib.Memory` whose store backend, `td.StoreBackend`, writes each result with the codec
registered for its type, the same registry records use:

- xarray results become zarr, DataFrames parquet, `{str: Tensor}` safetensors, and values of
  any registered codec use that codec; they are reopened lazily where the codec can;
- numpy arrays and plain Python values keep joblib's own pickle path (with memory mapping);
- records and views passed as arguments hash by their identity (type and key), not by their
  data.

In [ ]:
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
from jaxtyping import Float

import tensordir as td

root = Path(tempfile.mkdtemp(prefix="tensordir-memory-"))
td.set_root(root / "records")
memory = td.memory(root / "cache", verbose=0)  # other keywords go to joblib.Memory

## Results stored by codec

In [ ]:
calls = []


@memory.cache
def tuning_curve(preferred: float, width: float) -> xr.DataArray:
    calls.append("tuning_curve")
    theta = np.linspace(-np.pi, np.pi, 72, endpoint=False)
    return xr.DataArray(
        np.exp((np.cos(theta - preferred) - 1) / width), dims="orientation", coords={"orientation": theta}, name="rate"
    )


@memory.cache
def summary(widths: list[float]) -> pd.DataFrame:
    calls.append("summary")
    return pd.DataFrame({"width": widths, "bandwidth": [float((tuning_curve(0.0, w) > 0.5).mean()) for w in widths]})


@memory.cache
def noise(n: int) -> np.ndarray:
    calls.append("noise")
    return np.random.default_rng(0).normal(size=n)


curve = tuning_curve(0.5, 0.3)
table = summary([0.1, 0.3, 1.0])
samples = noise(1000)
print(calls)

A second call with the same arguments loads. The xarray result comes back opened lazily from
zarr, with its coordinates; the DataFrame from parquet; the numpy array through joblib.

In [ ]:
calls.clear()
print(type(tuning_curve(0.5, 0.3)).__name__, type(summary([0.1, 0.3, 1.0])).__name__, type(noise(1000)).__name__)
print("recomputed:", calls)
print(td.needs_compute(tuning_curve, 0.5, 0.3), td.needs_compute(tuning_curve, 0.5, 0.4))
summary([0.1, 0.3, 1.0])

On disk, codec-stored results live in `output.td/` next to joblib's `metadata.json`, with
`codec.json` naming the codec; the rest are joblib's `output.pkl`.

In [ ]:
for path in sorted((root / "cache").rglob("*")):
    if path.name in ("output.td", "output.pkl", "codec.json"):
        print("/".join(path.relative_to(root / "cache").parts[2:]))  # function / argument hash / ...

## Records as arguments

A record pickles as `(type, key, path)`, so joblib hashes which record it is, never its arrays.
Any handle to the same record hits the same cache entry.

In [ ]:
class Population(td.Record):
    preferred: Float[np.ndarray, "n_cells"]

    def __init__(self, /, *, n_cells: int = 200, seed: int = 0):
        self.preferred = np.random.default_rng(seed).uniform(-np.pi, np.pi, n_cells)


@memory.cache
def population_response(population: Population, stimulus: float) -> xr.DataArray:
    calls.append("population_response")
    rate = np.exp(np.cos(stimulus - population.preferred) - 1)
    return xr.DataArray(rate, dims="cell", name="rate")


calls.clear()
population_response(Population(), 0.0)
population_response(Population(n_cells=200), 0.0)  # another handle, same key: loaded
population_response(Population(seed=1), 0.0)  # another record: computed
print(calls)

Specs pickle in canonical form, so they hash by value: equal specs share an entry.

In [ ]:
@memory.cache
def n_cells_of(spec: Population.Spec) -> int:
    calls.append("n_cells_of")
    return spec.n_cells


calls.clear()
n_cells_of(Population.Spec()), n_cells_of(Population.Spec(n_cells=200)), n_cells_of(Population.Spec(seed=2))
print(calls)

## A cache per record

`record.memory` (and `view.memory`) is a joblib cache in the record's own `__cache__`
directory, so derived results travel and disappear with the record.

In [ ]:
def mean_rate(record, stimulus):
    return float(population_response(record, stimulus).mean())


population = Population()
cached_mean_rate = population.memory.cache(mean_rate, verbose=0)
print(round(cached_mean_rate(population, 1.0), 4))
population.tree()

## Choosing the storage

Two ways change what the backend does:

- **Register a codec.** Records and caches both pick it up, if the codec is faithful for the
  value (`codec.faithful(value)`: reading returns an equal value of the same type).
- **Subclass `td.StoreBackend`** and override `select_codec(item)` (return a codec, or `None`
  for joblib's pickle path) or `skip_codecs` (codec names never used; by default `ndarray`,
  `json` and `pickle`). Pass the class as `td.memory(location, backend=...)`.

In [ ]:
from tensordir.codecs import get_codec


class LargeArraysAsZarr(td.StoreBackend):
    """Arrays above 1 MB as zarr, reopened lazily as `zarr.Array`; the rest as usual."""

    def select_codec(self, item):
        if isinstance(item, np.ndarray) and item.nbytes > 2**20:
            return get_codec("ndarray")
        return super().select_codec(item)


class JsonToo(td.StoreBackend):
    skip_codecs = frozenset({"ndarray", "pickle"})  # plain Python values as JSON files


large = td.memory(root / "large", backend=LargeArraysAsZarr, verbose=0)
readable = td.memory(root / "readable", backend=JsonToo, verbose=0)

@large.cache
def activity(n_frames):
    return np.random.default_rng(0).normal(size=(n_frames, 1000))


@readable.cache
def settings(name):
    return {"name": name, "sampling_hz": 30}


activity(500), activity(5)
print(type(activity(500)).__name__, "(4 MB)", type(activity(5)).__name__, "(40 kB)")
settings("v1")
print(settings("v1"), [p.name for p in (root / "readable").rglob("output.json")])

`td.memory(location, lazy=False)` loads codec-stored results into memory instead of opening
them lazily.